# 第88课 · 教机器大海捞针——TF-IDF（词频×逆文档频率）检索，纯 NumPy 从零写起

**学习目标**
- 理解 TF-IDF 稀疏表示（Sparse Representation）的原理
- 从零实现 build_tfidf() 和 cosine_retrieve()
- 与 aurora.llm.retrieve 对比验证
- 用它检索一个小型 DSP 知识库

← **上一课**　[L87 · 量化与本地推理](L87_local_inference.ipynb)

> 上节课学习了 **量化与本地推理**：INT8 量化原理，连接 HuggingFace 轻量推理。  
> 本课将探讨 **TF-IDF 检索从零实现**。

## 谷歌搜索是怎么知道你在找"傅里叶变换"而不是"巴黎铁塔"的？

最简单的版本：**词频×稀有度**。

**TF（词频 Term Frequency）**：一篇文章里"傅里叶"出现了5次，它可能真的在讲傅里叶变换。

**IDF（逆文档频率 Inverse Document Frequency）**：
- "的"在所有文章里都有 → IDF 极低 → 权重极低
- "STFT"只在少数技术文章里有 → IDF 高 → 权重高

**组合**：`TF-IDF(词, 文档) = TF × IDF`

**直觉公式**：
```
TF(t, d)  = count(t, d) / |d|               → 词 t 在文档 d 中的相对频率
IDF(t)    = log((N+1)/(df(t)+1)) + 1         → N=文档总数, df=含该词的文档数
TF-IDF    = TF × IDF                         → 高频+稀有 → 高权重
```

**检索流程**：
```
查询 "Fourier transform"
  1. tokenize → ["fourier", "transform"]
  2. 构造查询向量 q（TF，不乘IDF）
  3. 计算 q 与每个文档向量的余弦相似度
  4. 返回 top-k 最相似文档
```

**为什么不直接用 sentence embeddings？** TF-IDF 不需要任何预训练权重，完全可以从公式推导，关键词匹配场景足够好，而且在 RAG 流水线里往往比向量检索更好解释。

本节实现 `tokenize` → `build_tfidf` → `cosine_retrieve`，与 `aurora.llm.retrieve` 对比验证。

### 🤔 IDF 里那个 log 是干嘛的？

公式里 `IDF = log((N+1)/(df+1)) + 1` 为什么要套一层 **log**？因为稀有度的价值是"边际递减"的：一个词从"出现在 100 篇"变成"10 篇"确实更有区分度，但从"10 篇"再变成"1 篇"没必要奖励一百倍。log 把 `N/df` 这个比值压扁，让常见词和稀有词的权重差距落在一个合理区间，而不是让极稀有词的权重直接爆炸。`+1` 平滑项则保证每个词至少有一点基础权重（避免 IDF=0 把整维抹掉）。

In [ ]:
import numpy as np
import re
from collections import Counter

In [ ]:
# 小型 DSP / 音频 AI 知识库（10 个文档）
DOCS = [
    'The Fourier transform decomposes a signal into its frequency components',
    'FFT stands for Fast Fourier Transform and runs in O N log N time',
    'The STFT applies a windowed Fourier transform to produce a spectrogram',
    'Mel filterbanks simulate human auditory perception of frequency',
    'MFCC Mel Frequency Cepstral Coefficients are computed via DCT after mel filterbank',
    'The Nyquist theorem states the sampling rate must be at least twice the maximum frequency',
    'Window functions like Hann and Hamming reduce spectral leakage in FFT',
    'CTC Connectionist Temporal Classification allows sequence labeling without alignment',
    'Whisper is a Transformer based speech recognition model from OpenAI',
    'Beat tracking estimates the tempo of music using onset detection and autocorrelation',
]
print(f'知识库：{len(DOCS)} 个文档')

## ✏️ 任务 1：实现 `tokenize`

**签名**：`tokenize(text: str) -> list[str]`

**2步实现路线**：

| 步骤 | 操作 | 代码提示 |
|---|---|---|
| 1 | 提取 ASCII 单词 | `re.findall(r'[a-z]+', ...)` |
| 2 | 转小写 | `.lower()` 后再 findall，或在模式里用 `re.I` flag |

**验收标准**：
- `tokenize('Hello World!')` == `['hello', 'world']`
- `tokenize('FFT O(N log N)')` == `['fft', 'o', 'n', 'log', 'n']`（括号和大写均正确处理）

In [ ]:
def tokenize(text: str) -> list:
    """简单 tokenizer：提取小写 ASCII 单词。"""
    # ✏️ TODO: 用正则提取单词，转小写
    raise NotImplementedError("TODO")

assert tokenize('Hello World!') == ['hello', 'world']
assert tokenize('FFT O(N log N)') == ['fft', 'o', 'n', 'log', 'n']
print('✅ tokenize 验证通过')

## ✏️ 任务 2：实现 `build_tfidf`

**签名**：`build_tfidf(docs: list) -> (matrix, vocab)`
- `matrix` shape: `(n_docs, vocab_size)`，值为 TF-IDF 权重
- `vocab`: 词汇表（Vocabulary，vocab）列表（已排序）

**3步实现路线**：

| 步骤 | 操作 | 公式 |
|---|---|---|
| 1 | 计算 TF 矩阵 | `TF[d, t] = count(t, d) / len(tokens_d)` |
| 2 | 计算 IDF 向量 | `IDF[t] = log((N+1) / (df(t)+1)) + 1`；`df(t)` = 含词 t 的文档数 |
| 3 | 返回 TF × IDF | `matrix = TF * IDF`（广播：`TF (n_docs, V)` × `IDF (V,)`） |

**验收标准**：
- `matrix.shape == (len(docs), len(vocab))`
- 高频+稀有词的权重最高（可打印 top 词验证）
- 与 `aurora.llm.build_tfidf` 数值完全一致（atol=1e-5）

In [ ]:
def build_tfidf(docs: list) -> tuple:
    """构建 TF-IDF 矩阵。返回 (matrix, vocab)。"""
    tokenized = [tokenize(d) for d in docs]
    all_terms = sorted({t for tokens in tokenized for t in tokens})
    word_idx = {w: i for i, w in enumerate(all_terms)}

    # ✏️ TODO:
    # (1) 计算 TF 矩阵 (n_docs, vocab_size)：TF(t,d) = count(t,d) / |d|
    # (2) 计算 IDF 向量 (vocab_size,)：IDF(t) = log((N+1)/(df(t)+1)) + 1
    #     df(t) = 含词 t 的文档数，N = 文档总数
    # (3) 返回 (TF × IDF, all_terms) — TF-IDF 矩阵和词汇表列表
    raise NotImplementedError("请实现 build_tfidf")

try:
    matrix, vocab = build_tfidf(DOCS)
    print(f'TF-IDF 矩阵 shape: {matrix.shape}  (n_docs={len(DOCS)}, vocab={len(vocab)})')
    print(f'词汇表前 10 词: {vocab[:10]}')
except (NotImplementedError, TypeError):
    matrix = None
    vocab = []
    print("⚠️  build_tfidf 尚未实现，matrix/vocab 已设为占位值。完成 TODO 后重新运行本格。")


## ✏️ 任务 3：实现 cosine_retrieve

**余弦相似度（Cosine Similarity）公式**
$$\cos(q,\,d) = \frac{q \cdot d}{\|q\|\,\|d\|}$$

- $q$ — 查询向量（TF，不乘 IDF）
- $d$ — 文档行向量（TF-IDF）
- 分母为零时（空查询或全零文档）得分记为 0

实现步骤：
1. 用查询词的 TF 构造 $q$（词不在词表中直接忽略）
2. 计算点积 $\text{scores}[i] = \text{tfidf\_matrix}[i] \cdot q$
3. 除以各自 L2 范数（零向量得 0 分）
4. 取 top-k 并按得分降序排列，返回 `[(doc, score), ...]`


In [ ]:
def cosine_retrieve(query: str, tfidf_matrix, vocab: list, docs: list, top_k=3) -> list:
    """余弦相似度检索，返回 (doc, score) 列表。"""
    word_idx = {w: i for i, w in enumerate(vocab)}
    tokens = tokenize(query)
    counts = Counter(tokens)
    total = max(len(tokens), 1)

    # ✏️ TODO:
    # (1) 构建查询向量 q_vec (vocab_size,)：q_vec[word_idx[t]] = counts[t] / total
    #     词不在 word_idx 中时忽略
    # (2) 计算点积：scores = tfidf_matrix @ q_vec  → shape (n_docs,)
    # (3) 余弦归一化：score_i /= (‖tfidf_matrix[i]‖ × ‖q_vec‖)
    #     若 ‖q_vec‖ == 0 或 ‖doc_vec‖ == 0，该文档得分置为 0
    # (4) 取 top_k：按得分降序排列，返回 [(docs[i], float(scores[i])), ...]
    raise NotImplementedError("请实现 cosine_retrieve")

# 测试（需先完成 build_tfidf 和 cosine_retrieve）
if matrix is not None:
    try:
        print('\n查询: "Fourier transform frequency"')
        for doc, score in cosine_retrieve('Fourier transform frequency', matrix, vocab, DOCS):
            print(f'  [{score:.3f}] {doc[:60]}')
        print('\n查询: "speech recognition neural network"')
        for doc, score in cosine_retrieve('speech recognition neural network', matrix, vocab, DOCS):
            print(f'  [{score:.3f}] {doc[:60]}')
    except (NotImplementedError, TypeError):
        print("⚠️  cosine_retrieve 尚未实现，完成 TODO 后重新运行本格。")
else:
    print("⚠️  matrix 为 None，请先完成 build_tfidf。")


## Aurora 连接

本课的两个函数对应 `src/aurora/llm/retrieve.py`：

| 本课实现 | 生产代码 |
|---|---|
| `build_tfidf(docs)` | `aurora.llm.build_tfidf` |
| `cosine_retrieve(query, ...)` | `aurora.llm.cosine_retrieve` |

下一课（L89）将直接 `from aurora.llm import ...` 把这个检索器接入完整的 RAG 流水线。


In [ ]:
# 与 aurora.llm 对比验证（需先完成两个 TODO）
from aurora.llm import build_tfidf as ref_build, cosine_retrieve as ref_retrieve

ref_mat, ref_vocab = ref_build(DOCS)
print(f'aurora.llm matrix shape: {ref_mat.shape}  ← 应与自实现相同')

if matrix is not None:
    # 1. 矩阵数值对比
    assert np.allclose(matrix, ref_mat, atol=1e-5), (
        f"❌ 矩阵与参考实现不一致！\n"
        f"  自实现 shape={matrix.shape}, max={matrix.max():.4f}\n"
        f"  参考   shape={ref_mat.shape}, max={ref_mat.max():.4f}\n"
        "检查你的 TF 公式和 IDF 公式是否与 cell 1 一致。"
    )
    print("✅ build_tfidf 矩阵与参考实现一致")

    # 2. 检索顺序对比
    try:
        my_results = cosine_retrieve('Fourier transform frequency', matrix, vocab, DOCS)
        ref_results = ref_retrieve('Fourier transform frequency', ref_mat, ref_vocab, DOCS)
        my_docs = [d for d, _ in my_results]
        ref_docs = [d for d, _ in ref_results]
        assert my_docs == ref_docs, (
            f"❌ 检索顺序与参考实现不一致！\n"
            f"  自实现: {my_docs}\n"
            f"  参考:   {ref_docs}"
        )
        print("✅ cosine_retrieve 结果顺序与参考实现一致")
        print("✅ TF-IDF 检索验证通过")
    except (NotImplementedError, TypeError):
        print("⚠️  cosine_retrieve 尚未实现，跳过检索顺序对比。")
else:
    # 仅显示参考结果供参考
    ref_results = ref_retrieve('Fourier transform frequency', ref_mat, ref_vocab, DOCS)
    print("aurora.llm 检索结果（参考）:", [s for _, s in ref_results])
    print("⚠️  build_tfidf 尚未实现，跳过自实现对比。")


## ✏️ 闭卷推导检查格 — TF-IDF 手算

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：有 3 个文档，词汇表只有 4 个词：`{cat, dog, the, run}`

| 文档 | 内容 |
|---|---|
| D0 | "the cat run" |
| D1 | "the dog run run" |
| D2 | "cat cat dog" |

1. 计算每个词的 `df`（含该词的文档数）
2. 计算 `IDF`（用公式 `log((3+1)/(df+1)) + 1`）
3. 计算 D1 的 TF 向量
4. 计算 D1 的 TF-IDF 向量
5. 查询 "cat run" 的余弦相似度与哪个文档最高？（手算或估算）

（在此处写推导...）

In [ ]:
# ✏️ 本课自评
l88_review = {
    "tfidf_intuition":           None,  # 理解TF=词频，IDF=稀有度，组合=高频+稀有→高权重？True/False
    "tokenize_impl":             None,  # tokenize()实现正确，两个assert通过？True/False
    "build_tfidf_impl":          None,  # build_tfidf()实现正确，与aurora.llm一致？True/False
    "cosine_retrieve_impl":      None,  # cosine_retrieve()实现正确，检索顺序对齐？True/False
    "whiteboard_passed":         None,  # 白板挑战TF-IDF手算推导通关？True/False
}

unfilled = [k for k, v in l88_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l88_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L88 全部通关！进入 L89：RAG 完整流水线')

In [ ]:
# 验证：手算 TF-IDF 数值（3文档×4词小例子）
import numpy as np

MINI_DOCS = ["the cat run", "the dog run run", "cat cat dog"]
MINI_VOCAB = ["cat", "dog", "run", "the"]  # sorted

# 手动 TF
tf_manual = np.array([
    [1/3, 0,   1/3, 1/3],   # D0: cat=1/3, dog=0, run=1/3, the=1/3
    [0,   1/4, 2/4, 1/4],   # D1: cat=0, dog=1/4, run=2/4, the=1/4
    [2/3, 1/3, 0,   0  ],   # D2: cat=2/3, dog=1/3, run=0, the=0
])

# df per word: cat in D0,D2 → df=2; dog in D1,D2 → df=2; run in D0,D1 → df=2; the in D0,D1 → df=2
df = np.array([2, 2, 2, 2])
N = 3
idf_manual = np.log((N + 1) / (df + 1)) + 1   # smoothed

tfidf_manual = tf_manual * idf_manual

# D0 TF-IDF
assert tfidf_manual.shape == (3, 4), f"shape应(3,4)，得{tfidf_manual.shape}"
assert tfidf_manual[0, 0] > 0, "D0 cat 权重应>0"

# 对角线自洽：稀有词权重=TF×IDF，常见词IDF低
print("IDF per word (cat/dog/run/the):", [f"{v:.4f}" for v in idf_manual])
print("D1 TF-IDF:", [f"{v:.4f}" for v in tfidf_manual[1]])

# 验证 build_tfidf 结果（若已实现）
try:
    matrix, vocab = build_tfidf(MINI_DOCS)
    assert np.allclose(matrix, tfidf_manual, atol=1e-9), \
        f"与手算结果不一致！\n自实现:\n{matrix}\n手算:\n{tfidf_manual}"
    print("✅ build_tfidf 与手算完全吻合")
except (NotImplementedError, TypeError):
    print("⚠️ build_tfidf 尚未实现，手算验证已完成")
except Exception as e:
    print(f"⚠️ 验证异常（build_tfidf 尚未完成）: {e}")

## 本课收束

| 概念 | 要记住的 |
|---|---|
| TF | 词在文档中的相对频率 |
| IDF | 含这个词的文档越多，权重越低（常见词权重低）|
| 余弦相似度 | 方向相同 → 相似，与向量长度无关 |
| 优点 | 无需预训练，可完全理解，快速 O(n×d) |
| 缺点 | 语义匹配弱（"car"≠"automobile"），无上下文理解 |
| L89 | 把这个检索器组装进完整的 RAG 流水线 |

下一课（L89）把 TF-IDF 检索、向量相似度和 LLM 生成组合成完整的 RAG 管道（RAG pipeline），端到端回答问题。

---

→ **下一课**　[L89 · RAG 完整流水线](L89_rag_pipeline.ipynb)

> 下节课将学习 **RAG 完整流水线**：文档切片 + TF-IDF 检索 + 提示词构建 + 生成。